In [54]:
import pandas as pd

cols = ["id", "entity", "sentiment", "text"]#naming the columns so the first row will not be considered as the name of the columns

train = pd.read_csv("../data/twitter_training.csv", header=None)
train.columns = ["id", "entity", "sentiment", "text"]
val   = pd.read_csv("../data/twitter_validation.csv", header=None)
val.columns = ["id", "entity", "sentiment", "text"]



In [55]:
train.head()

,id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [56]:
val.head()

,id,entity,sentiment,text
0,3364,Facebook,Irrelevant,I mentioned on Facebook that I was struggling ...
1,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
3,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
4,4433,Google,Neutral,Now the President is slapping Americans in the...


In [57]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         74682 non-null  int64
 1   entity     74682 non-null  str  
 2   sentiment  74682 non-null  str  
 3   text       73996 non-null  str  
dtypes: int64(1), str(3)
memory usage: 2.3 MB


In [58]:
print("TRAIN:", train.shape, "VAL:", val.shape)

TRAIN: (74682, 4) VAL: (1000, 4)


In [59]:
print("Missing text train:", train["text"].isna().sum())
print("Missing text val:", val["text"].isna().sum())

Missing text train: 686
Missing text val: 0


In [60]:
print(train["sentiment"].value_counts())

sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [61]:
train = train.dropna(subset=["text"]).copy()

In [62]:
print("Missing text train:", train["text"].isna().sum())

Missing text train: 0


In [63]:
# check if the Same person (id) wrote the exact same text, but it appears in the dataset with different sentiment labels. (duplicates)
conflicts = (
    train.groupby(["id", "text"])["sentiment"] #splits the dataset into pairs and select the sentiment
       .nunique() #number of unique values
       .reset_index(name="unique_sentiments")#turns it into a normal table and names the count column unique_sentiments.
)

# rows where same person + same text has more than 1 sentiment
conflicts = conflicts[conflicts["unique_sentiments"] > 1] #This filters to only those (id, text) pairs where there is more than 1 unique sentiment

print("Conflicting (id, text) pairs:", len(conflicts))
conflicts.head()

Conflicting (id, text) pairs: 0


,id,text,unique_sentiments


In [64]:
#Converts to string and removes whitespace at the start and end
train["text"] = train["text"].astype(str)
train["entity"] = train["entity"].astype(str).str.strip()
train["sentiment"] = train["sentiment"].astype(str).str.strip()

In [65]:
print("Exact duplicate rows:", train.duplicated().sum())



Exact duplicate rows: 2340


In [66]:
train_clean = train.drop_duplicates().copy()
print("Exact duplicate rows:", train.duplicated().sum())


Exact duplicate rows: 2340


In [67]:

dup_id_text = train_clean.duplicated(subset=["id", "text"], keep=False).sum()
print("Duplicate (id,text) rows:", dup_id_text)

Duplicate (id,text) rows: 0


In [70]:
train_clean.to_csv("../data/train_clean.csv", index=False)
val.to_csv("../data/val_clean.csv", index=False)